# HotPotQA ReAct - LangChain `create_agent` 버전

`create_agent`를 사용하여 ReAct 에이전트를 간결하게 구현한다.  
에이전트가 자동으로 Thought → Action → Observation 루프를 수행한다.

## 1. Setup

In [1]:
import os
import sys

from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from tools import search, lookup, finish, wiki_state
from eval_utils import load_hotpotqa, normalize_answer, f1_score, get_metrics

## 2. 모델 및 에이전트 생성

In [2]:
model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    max_tokens=1000,
)

system_prompt = """Thought, Action, Observation 단계를 번갈아가며 수행하여 질문 응답 과제를 해결하시오.

Thought는 현재 상황에 대해 추론하는 단계이며,
Action은 다음 세 가지 도구 중 하나를 사용해야 한다:

(1) search: Wikipedia에서 해당 entity를 검색하고, 존재할 경우 첫 번째 문단을 반환한다.
(2) lookup: 현재 문서에서 해당 keyword가 포함된 다음 문장을 반환한다.
(3) finish: 최종 답을 반환하고 과제를 종료한다. 답을 확정했을 때 반드시 이 도구를 호출하시오.

규칙:
- 각 단계에서 반드시 Thought를 먼저 수행한 뒤 도구를 호출하시오.
- 최대 7단계 이내에 답을 제출하시오.
- 최종 답은 간결하게 (단어 또는 짧은 구문) 제출하시오.
- 반드시 finish 도구를 호출하여 답을 제출하시오.
"""

agent = create_agent(
    model=model,
    tools=[search, lookup, finish],
    system_prompt=system_prompt,
)

print("Agent created successfully.")

Agent created successfully.


## 3. 실행 함수 정의

In [3]:
def extract_answer(messages) -> str:
    """에이전트 응답에서 최종 답을 추출한다. finish tool 호출의 인자에서 답을 가져온다."""
    # 1) finish tool 호출에서 답 추출 (가장 신뢰)
    for msg in reversed(messages):
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                if tc["name"] == "finish":
                    return tc["args"].get("answer", "")
    # 2) finish tool 반환 메시지에서 추출
    for msg in reversed(messages):
        content = msg.content if hasattr(msg, "content") else str(msg)
        if "Episode finished, answer:" in content:
            return content.split("Episode finished, answer:")[-1].strip()
    # 3) fallback: 마지막 AI 메시지
    for msg in reversed(messages):
        if hasattr(msg, "content") and msg.type == "ai" and msg.content:
            return msg.content.strip()
    return ""


def run_hotpotqa(question: str, to_print=True) -> tuple[str, list]:
    """단일 질문에 대해 에이전트를 실행한다."""
    wiki_state.reset()

    result = agent.invoke({
        "messages": [{"role": "user", "content": f"Question: {question}"}]
    })

    messages = result["messages"]
    answer = extract_answer(messages)

    if to_print:
        print(f"Q: {question}")
        for msg in messages[1:]:  # system 제외
            role = msg.type if hasattr(msg, "type") else "unknown"
            content = msg.content if hasattr(msg, "content") else str(msg)
            if content:
                print(f"  [{role}] {content[:200]}")
        print(f"  => Answer: {answer}")

    return answer, messages

## 4. 단일 질문 테스트

In [4]:
data = load_hotpotqa("dev")
print(f"Loaded {len(data)} questions.")

# 단일 테스트
question, gt_answer = data[0]
pred, _ = run_hotpotqa(question)
print(f"\nGT: {gt_answer}")
print(f"Pred: {pred}")
print(f"EM: {normalize_answer(pred) == normalize_answer(gt_answer)}")

Loaded 7405 questions.
Q: Were Scott Derrickson and Ed Wood of the same nationality?
  [ai] Thought: To answer whether Scott Derrickson and Ed Wood share the same nationality, I need to find out the nationality of each person. I will search for both Scott Derrickson and Ed Wood on Wikipedia 
  [tool] Scott Derrickson (born July 16, 1966) is an American filmmaker. He is known for his work in the horror genre, directing films such as The Exorcism of Emily Rose (2005), Sinister (2012), The Black Phon
  [tool] Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, and pulp novelist.. In the 1950s, Wood directed several low-budget science fiction, crime and horror film
  [ai] Thought: Both Scott Derrickson and Ed Wood are described as American filmmakers in their respective Wikipedia summaries. This indicates that they share the same nationality, which is American.

Action
  [tool] Episode finished, answer: Yes, both are American
  [ai] Yes, both are 

## 6. 배치 평가 (500개 샘플)

In [5]:
import random
import time

idxs = list(range(len(data)))
random.Random(233).shuffle(idxs)

results = []
old_time = time.time()

for i, idx in enumerate(idxs[:500]):
    question, gt_answer = data[idx]
    try:
        pred, _ = run_hotpotqa(question, to_print=True)
    except Exception as e:
        print(f"Error at idx {idx}: {e}")
        pred = ""

    metrics = get_metrics(pred, gt_answer)
    results.append(metrics)

    em_sum = sum(r["em"] for r in results)
    avg_em = em_sum / len(results)
    elapsed = (time.time() - old_time) / len(results)
    print(f"[{len(results)}/500] EM={em_sum}/{len(results)} ({avg_em:.3f}) | {elapsed:.1f}s/q")
    print(f"GT: {gt_answer}")
    print(f"Pred: {pred}")
    print("-" * 40)

# 최종 결과
total_em = sum(r["em"] for r in results)
total_f1 = sum(r["f1"] for r in results)
print(f"\n=== Final Results ===")
print(f"EM: {total_em}/{len(results)} ({total_em/len(results):.3f})")
print(f"F1: {total_f1/len(results):.3f}")

Q: What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?
  [ai] Thought: The question asks about a movie that actress Irene Jacob completed before an American action crime thriller film directed by Stuart Bird. First, I need to identify the American action crime t
  [tool] Could not find Stuart Bird. Similar: ['Stuart Little', 'State Bird Provisions', 'Stuart Keith', 'The Good Lord Bird (miniseries)', 'Flappy Bird'].
  [ai] Thought: There is no direct information about Stuart Bird. I will try searching for "American action crime thriller film directed by Stuart Bird" to see if there is a film associated with this directo
  [tool] Could not find American action crime thriller film directed by Stuart Bird. Similar: ['List of American films of 2024', 'List of action films of the 2020s', 'List of American films of 2023', "Dylan O'
  [ai] Thought: I could not find direct information about the film directed by Stuart Bird. I will 

KeyboardInterrupt: 